# Démonstration d'inférence U-TILISE

Ce notebook charge un modèle entraîné, produit des reconstructions sans nuages
et compare deux stratégies de masquage :

- **Random Clouds** : masques nuageux partiels échantillonnés aléatoirement
- **Random Fully Masked** : dates claires masquées entièrement de façon aléatoire

> **Prérequis :** l'environnement conda `cloud_reconstruction` doit être activé.
> Un checkpoint entraîné et le fichier HDF5 doivent être accessibles.

In [ ]:
import os
import sys
import warnings

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore', category=FutureWarning)
%matplotlib inline

# Se placer à la racine du projet
_cwd = os.path.abspath(os.getcwd())
if os.path.exists(os.path.join(_cwd, 'configs', 'default.yaml')):
    PROJECT_ROOT = _cwd
elif os.path.exists(os.path.join(_cwd, '..', 'configs', 'default.yaml')):
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, '..'))
else:
    raise FileNotFoundError('Impossible de trouver la racine du projet')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()

from omegaconf import OmegaConf
from copy import deepcopy
from src import config_utils, data_utils, visutils
from src.data_utils import extract_sample
from src.eval_tools import Imputation

print(f'Répertoire de travail : {PROJECT_ROOT}')

## 1. Paramètres

In [ ]:
# Les chemins sont lus depuis le fichier .env (voir .env.example)
CHECKPOINT = os.environ['CHECKPOINT']
CONFIG_TRAIN = os.environ['TRAIN_CONFIG']
HDF5_FILE = os.environ['HDF5_FILE']

# Paramètres d'inférence
TEMPORAL_WINDOW = 14
BLEND_MODE = 'center'
SAMPLE_IDX = 1250  # ← Modifier cet indice pour changer de patch

print(f'Checkpoint      : {CHECKPOINT}')
print(f'Config train    : {CONFIG_TRAIN}')
print(f'HDF5            : {HDF5_FILE}')
print(f'Fenêtre         : {TEMPORAL_WINDOW}')
print(f'Mode de fusion  : {BLEND_MODE}')
print(f'Sample          : {SAMPLE_IDX}')

## 2. Fonctions utilitaires

In [ ]:
def _to_cpu(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().clone()
    elif isinstance(x, dict):
        return {k: _to_cpu(v) for k, v in x.items()}
    elif isinstance(x, (list, tuple)):
        return type(x)(_to_cpu(v) for v in x)
    return x


def _draw_grid(images, ncols, column_labels, row_labels, ax_array, fig,
               BRIGHTNESS_FACTOR, show_col_labels=True):
    images = visutils.apply_brightness_factor(images, factor=BRIGHTNESS_FACTOR)
    for idx, ax in enumerate(ax_array.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        rect = mpatches.Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                  fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        if show_col_labels and idx < ncols:
            ax.set_title(column_labels[idx], fontsize=9, fontweight='bold',
                         rotation=45, ha='left')
    for i, label in enumerate(row_labels):
        ax_array[i, 0].annotate(label, xy=(-0.1, 0.5), xycoords='axes fraction',
                                fontsize=10, fontweight='bold',
                                ha='right', va='center', rotation=90)


def plot_seq_RGB_NIR_interactive(targets, inputs, preds, idx_rgb, idx_nir=None,
                                 n_visible=8, title='', dates=None):
    """Slider interactif RGB + NIR-R-G."""
    targets = _to_cpu(targets)
    inputs  = _to_cpu(inputs)
    preds   = _to_cpu(preds)

    target_rgb = targets[:, idx_rgb].swapaxes(1, 3).swapaxes(1, 2)
    inputs_rgb = inputs[:, idx_rgb].swapaxes(1, 3).swapaxes(1, 2)
    preds_rgb  = preds[:, idx_rgb].swapaxes(1, 3).swapaxes(1, 2)

    has_nir = idx_nir is not None
    if has_nir:
        idx_nir_rg = [idx_nir] + list(idx_rgb[:2])
        target_nir = targets[:, idx_nir_rg].swapaxes(1, 3).swapaxes(1, 2)
        inputs_nir = inputs[:, idx_nir_rg].swapaxes(1, 3).swapaxes(1, 2)
        preds_nir  = preds[:, idx_nir_rg].swapaxes(1, 3).swapaxes(1, 2)

    T = target_rgb.shape[0]
    n_visible = min(n_visible, T)
    if dates is None:
        dates = [f't{i}' for i in range(T)]

    total_rows = 6 if has_nir else 3

    out = widgets.Output()
    slider = widgets.IntSlider(
        value=0, min=0, max=max(T - n_visible, 0), step=1,
        description='t_start :', continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'),
    )
    label = widgets.Label(
        value=f'{title}  —  Série : {T} dates  |  Fenêtre : {n_visible}'
    )

    def _update(_change=None):
        t0 = slider.value
        idx = slice(t0, t0 + n_visible)
        col_labels = dates[t0:t0 + n_visible]
        imgs_rgb = torch.cat([target_rgb[idx], inputs_rgb[idx], preds_rgb[idx]], dim=0)
        row_labels = ['Target', 'Inputs', 'Predictions']
        with out:
            clear_output(wait=True)
            fig, axes = plt.subplots(nrows=total_rows, ncols=n_visible,
                                     figsize=(2.2 * n_visible, 2.2 * total_rows))
            _draw_grid(imgs_rgb, n_visible, col_labels, row_labels,
                       axes[:3], fig, BRIGHTNESS_FACTOR=3, show_col_labels=True)
            if has_nir:
                imgs_nir = torch.cat([target_nir[idx], inputs_nir[idx], preds_nir[idx]], dim=0)
                _draw_grid(imgs_nir, n_visible, col_labels, row_labels,
                           axes[3:], fig, BRIGHTNESS_FACTOR=2, show_col_labels=False)
                fig.text(0.02, 0.78, 'RGB', fontsize=14, fontweight='bold',
                         rotation=90, va='center')
                fig.text(0.02, 0.35, 'NIR-R-G', fontsize=14, fontweight='bold',
                         rotation=90, va='center')
            plt.subplots_adjust(left=0.08, top=0.92, wspace=0.05, hspace=0.15)
            plt.show()

    slider.observe(_update, names='value')
    display(widgets.VBox([label, slider, out]))
    _update()

## 3. Chargement du modèle

In [ ]:
# Charger la config et préparer un dataset de référence
cfg_default = config_utils.read_config(os.path.join(PROJECT_ROOT, 'configs/default.yaml'))
cfg_eval = config_utils.read_config(os.path.join(PROJECT_ROOT, 'configs/config_run_eval.yaml'))
config = OmegaConf.merge(cfg_default, cfg_eval)
config.data.hdf5_file = HDF5_FILE

# Dataset de référence pour obtenir num_channels et les indices de bandes
test_dset_ref = data_utils.get_dataset(config, phase='test')
num_channels = test_dset_ref.num_channels
idx_rgb = test_dset_ref.c_index_rgb.int().tolist() if isinstance(test_dset_ref.c_index_rgb, torch.Tensor) else test_dset_ref.c_index_rgb
idx_nir = test_dset_ref.c_index_nir
if isinstance(idx_nir, torch.Tensor):
    idx_nir = idx_nir.item()
if np.isnan(idx_nir):
    idx_nir = None
else:
    idx_nir = int(idx_nir)

print(f'Nombre de canaux : {num_channels}')
print(f'Indices RGB      : {idx_rgb}')
print(f'Indice NIR       : {idx_nir}')
print(f'Échantillons     : {len(test_dset_ref)}')

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

imputer = Imputation(
    config_file_train=CONFIG_TRAIN,
    checkpoint=CHECKPOINT,
    temporal_window=TEMPORAL_WINDOW,
    blend_mode=BLEND_MODE,
    num_channels=num_channels,
    device=device,
)

## 4. Random Clouds (masquage spatial partiel)

Des masques nuageux sont échantillonnés aléatoirement depuis d'autres tuiles
et appliqués spatialement sur certaines dates. Le masquage est **partiel** :
seule une partie de l'image est masquée à chaque date.

In [ ]:
# Créer le dataset avec mask_type = random_clouds
cfg_rc = deepcopy(config)
cfg_rc.mask.mask_type = 'random_clouds'
dset_rc = data_utils.get_dataset(cfg_rc, phase='test')

# Charger l'échantillon et lancer l'inférence
sample_rc = dset_rc[SAMPLE_IDX]
batch_rc = {k: v.unsqueeze(0) if isinstance(v, torch.Tensor) else v for k, v in sample_rc.items()}
batch_rc, y_pred_rc = imputer.impute_sample(batch_rc)

print(f'Prédiction : {y_pred_rc.shape}')
print(f'Dates      : {batch_rc.get("S2_dates", "N/A")}')

In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_rc['y'][0],
    inputs=batch_rc['x'][0],
    preds=y_pred_rc[0],
    idx_rgb=idx_rgb,
    idx_nir=idx_nir,
    n_visible=8,
    title=f'Random Clouds — échantillon {SAMPLE_IDX}',
    dates=batch_rc.get('S2_dates'),
)

## 5. Random Fully Masked (dates aléatoires entièrement masquées)

Des dates claires sont choisies aléatoirement et **entièrement masquées**.
Le modèle doit les reconstruire sans aucune information optique sur ces dates.

In [ ]:
# Créer le dataset avec mask_type = random_fully_masked
cfg_rfm = deepcopy(config)
cfg_rfm.mask.mask_type = 'random_fully_masked'
dset_rfm = data_utils.get_dataset(cfg_rfm, phase='test')

# Charger l'échantillon et lancer l'inférence
sample_rfm = dset_rfm[SAMPLE_IDX]
batch_rfm = {k: v.unsqueeze(0) if isinstance(v, torch.Tensor) else v for k, v in sample_rfm.items()}
batch_rfm, y_pred_rfm = imputer.impute_sample(batch_rfm)

print(f'Prédiction : {y_pred_rfm.shape}')
print(f'Dates      : {batch_rfm.get("S2_dates", "N/A")}')

In [ ]:
plot_seq_RGB_NIR_interactive(
    targets=batch_rfm['y'][0],
    inputs=batch_rfm['x'][0],
    preds=y_pred_rfm[0],
    idx_rgb=idx_rgb,
    idx_nir=idx_nir,
    n_visible=8,
    title=f'Random Fully Masked — échantillon {SAMPLE_IDX}',
    dates=batch_rfm.get('S2_dates'),
)

## 6. Métriques de reconstruction

Comparaison des métriques pour les deux stratégies de masquage.

In [ ]:
from src.metrics.cloud_removal import CloudRemovalMetrics

compute_metrics = CloudRemovalMetrics(
    metrics=['mae', 'mse', 'rmse', 'psnr', 'ssim', 'sam'],
    eval_occluded_observed=True,
)

results = {
    'Random Clouds': (batch_rc, y_pred_rc),
    'Random Fully Masked': (batch_rfm, y_pred_rfm),
}

for label, (batch, y_pred) in results.items():
    inputs, target, masks, mask_valid, cloud_mask, _, _ = extract_sample(batch)
    m = compute_metrics(target, masks, y_pred, cloud_mask)
    print(f'\n=== {label} ===')
    for name, value in m.items():
        v = value.item() if hasattr(value, 'item') else value
        if isinstance(v, (float, int)):
            print(f'  {name:30s} : {v:.4f}')